In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
apps_df=pd.read_csv('../data/Play Store Data.csv')
reviews_df=pd.read_csv('../data/User Reviews.csv')


In [4]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [6]:
#pd.read_csv() : csv files
#pd.read_excel() : excel files
#pd.read_sql() : Sql Databses
#pd.read_json() : JSON Files

In [7]:
#df.isnull() : Missing Values
#df.dropna() : Removes rows and columns that contain the missing values
#df.fillna() : Fills missing values

In [8]:
#df.duplicate() : Identifies duplicates
#df.drop_duplicates() : Removes duplicate rows

In [9]:
import pandas as pd
apps_df=pd.read_csv('../data/Play Store Data.csv')
reviews_df=pd.read_csv('../data/User Reviews.csv')

#Step 2 : Data Cleaning
apps_df =apps_df.dropna(subset=['Rating'])
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)
apps_df.drop_duplicates(inplace=True)
apps_df=apps_df[apps_df['Rating']<=5]
reviews_df.dropna(subset=['Translated_Review'],inplace=True)

/tmp/ipykernel_660/2769953446.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)


In [10]:
# Covert the installs columns to numeric by removing comas and +
apps_df['Installs']=apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)

#Convert Price column to numeric after removing $
apps_df['Price']=apps_df['Price'].str.replace('$','').astype(float)



In [11]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs            int64
Type                  str
Price             float64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

In [12]:
#merging the data tables
merged_df=pd.merge(apps_df,reviews_df,on='App',how='inner')
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [13]:
#Data transformations

def convert_size(size):
  if 'M' in size:
    return float(size.replace('M',''))
  elif 'k' in size:
    return float(size.replace('k',''))/1024
  else:
    return np.nan
apps_df['Size']=apps_df['Size'].apply(convert_size)


In [14]:
apps_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10834,FR Calculator,FAMILY,4.0,7,2.6,500,Free,0.0,Everyone,Education,"June 18, 2017",1.0.0,4.1 and up
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53.0,5000,Free,0.0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6,100,Free,0.0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,NaN,1000,Free,0.0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [15]:
#logarithmic transformation

apps_df['Log_Installs']=np.log(apps_df['Installs'])
#apps_df['Log_Reviews']=np.log(apps_df['Reviews'])
# as the error occured because reviews is still an object nd not int


In [16]:
apps_df['Reviews']=apps_df['Reviews'].astype(int)

In [17]:
apps_df['Log_Reviews']=np.log(apps_df['Reviews'])

In [18]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews             int64
Size              float64
Installs            int64
Type                  str
Price             float64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
Log_Installs      float64
Log_Reviews       float64
dtype: object

In [19]:
def rating_group(rating):
  if rating >=4:
    return 'Top rated app'
  elif rating >=3:
    return 'Above average'
  elif rating >=2:
    return 'Average'
  else:
    return 'Below average'
apps_df['Rating_Group']=apps_df['Rating'].apply(rating_group)



In [20]:
#Revenue column
apps_df['Revenue']=apps_df['Price']*apps_df['Installs']

In [21]:
#SENTIMENT ANALYSIS NLP
sia=SentimentIntensityAnalyzer()


In [22]:
#polarity score in SIA
# positive, negative, neutral and coumpound: -1- very negative; +1- very positive

In [23]:
review="This app is amazing. I love the new feature"
sentiment_score=sia.polarity_scores(review)
print(sentiment_score)


{'neg': 0.0, 'neu': 0.429, 'pos': 0.571, 'compound': 0.8402}


In [24]:
review="This app is bad. I hated the new feature"
sentiment_score=sia.polarity_scores(review)
print(sentiment_score)


{'neg': 0.562, 'neu': 0.438, 'pos': 0.0, 'compound': -0.8271}


In [25]:
review="This app is good. features can be improved"
sentiment_score=sia.polarity_scores(review)
print(sentiment_score)


{'neg': 0.0, 'neu': 0.5, 'pos': 0.5, 'compound': 0.7184}


In [26]:
reviews_df['Sentiment_Score'] = reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x)).get('compound', 0.0))
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [27]:
apps_df['Last Updated']=pd.to_datetime(apps_df['Last Updated'],errors='coerce')

In [28]:
apps_df['Year']=apps_df['Last Updated'].dt.year

In [29]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122363,6.874198,Above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,2018


In [30]:
html_files_paths="./"
if not os.path.exists(html_files_paths):
  os.makedirs(html_files_paths)

In [31]:
plot_containers=""
def save_plot_as_html(fig,filename,insights):
  global plot_containers
  filepath=os.path.join(html_files_paths,filename)
  html_content=pio.to_html(fig,full_html=False,include_plotlyjs='inline')
  plot_containers+=f"""<div class ="plot-container" id='{filename}' onclick="openPlot('{filename}')">
  <div class="plot"> {html_content}</div> <div class='insights'>{insights}</div> </div>"""
  fig.write_html(filepath,full_html=False,include_plotlyjs='inline')

In [32]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [33]:
#Fig1
category_count=apps_df['Category'].value_counts().nlargest(10)
fig1=px.bar(
    x=category_count.index,
    y=category_count.values,
    labels={'x':'Category','y':'Count'},
    title='Top Categories on Play Store',
    color=category_count.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,height=300

)
fig1.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig1.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig1,"Category Graph 1.html","The top categories on the play store are.")

In [34]:
#Fig2 Type analysis
type_count=apps_df['Type'].value_counts()
fig2=px.pie(
    values=type_count.values,
    names=type_count.index,
    title='Type Analysis',
    color=type_count.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,height=300
)



fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig1.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig2,"Type Graph 1.html","Most apps on the playstore are free,indicating a strategy to attract users first and monetize through ads.")

In [35]:
#fig3 rating distribution

fig3=px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=[px.colors.sequential.Plasma[5]],
    labels={'Rating':'Rating'},
    width=400,height=300

)



fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig1.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig3,"Rating Graph 3.html","Ratings are skewed towards higher values, suggesting that most apps are rated favourably by users.")


In [36]:
#fig 4
sentiment_count_df=reviews_df['Sentiment'].value_counts()
fig4=px.bar(
    x=sentiment_count_df.index,
    y=sentiment_count_df.values,
    labels={'x':'Sentiment Score','y':'Count'},
    title='Sentiment Distribution',
    color=sentiment_count_df.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=400,height=300


)



fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig4.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig4,"Sentiment Graph 4.html","Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments.")




In [37]:
# Fig5
installs_by_category=apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x':'Total Installs','y':'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,height=300
)
fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig5, "Installs Graph 5.html","The categories with the most installs are social and communication apps, reflecting their broad appeal.")

In [38]:
#fig6
updates_per_year=apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6=px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x':'Year','y':'Count'},
    title='Updates per Year',
    color_discrete_sequence=['#AB63FA'],
    width=400,height=300
)
fig6.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(fig6,"updates_per_year.html", "Updates have been increasing over the years, showing that developers are actively maintaining and import. ")



In [39]:
import plotly.express as px
#fig7
revenue_by_category=apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=revenue_by_category.index,
    y=revenue_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=revenue_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,height=300
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(fig7,"Revenue graph 7.html", "Categories such as Business and Productivity lead in revenue generate, indicating their monetization ")

In [40]:
#fig8

genre_counts=apps_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top genres',
    color=revenue_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,height=300
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(fig8,"Genres graph 8.html", "Action and Casual genres are the most common, reflecting user preferencees for action, and easy to play games. ")

In [41]:
#Fig9
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [42]:
#Figure 10 for identifing outliners
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")


In [43]:
plot_containers_split=plot_containers.split('</div>')

In [44]:
if len(plot_containers_split)>1:
  final_plot=plot_containers_split[-2]+'</div>'
else:
  final_plot=plot_containers

In [45]:
# Dashboard Creation
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="https://images.seeklogo.com/logo-png/62/1/google-new-logo-png_seeklogo-622426.png" alt="Google Logo">
            <h1>Google Play Store Reviews Analytics</h1>
            <img src="https://www.gstatic.com/marketing-cms/assets/images/15/b9/77649f194169be94fc4631a785bc/play-symbol.webp=n-w963-h543-fcrop64=1,380c0000c841ffff-rw" alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """


In [46]:
dashboard_full_html = dashboard_html.format(plot_width=plot_width, plot_height=plot_height, plots=plot_containers)

with open('dashboard.html', 'w') as f:
    f.write(dashboard_full_html)

print('Dashboard HTML saved to dashboard.html. You can download this file and open it in a new browser tab.')

Dashboard HTML saved to dashboard.html. You can download this file and open it in a new browser tab.


In [47]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [48]:
dashboard_path=os.path.join(html_files_paths,"web page.html")

In [49]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

In [50]:
# webbrowser.open(...) skipped when running headless/outside a desktop environment


In [51]:
from IPython.display import HTML

# Assuming dashboard_full_html contains the complete HTML for your dashboard
# If not, you can read it from the saved file:
# with open('dashboard.html', 'r') as f:
#     dashboard_content = f.read()
# display(HTML(dashboard_content))

print('Dashboard assembled - open output/web page.html (or dashboard.html) in a browser to view it. Not displaying inline here to keep the notebook file small.')


Dashboard assembled - open output/web page.html (or dashboard.html) in a browser to view it. Not displaying inline here to keep the notebook file small.


Task1

In [52]:
import datetime
import pytz

# Step 1: Filter — Jan updates, rating >= 4.0, size >= 10MB
filtered_apps_df = apps_df[
    (apps_df['Last Updated'].dt.month == 1) &
    (apps_df['Rating'] >= 4.0) &
    (apps_df['Size'].notna()) &
    (apps_df['Size'] >= 10)
]

# Top 10 categories by installs (from filtered data)
top_10_categories_by_installs = (
    filtered_apps_df.groupby('Category')['Installs'].sum().nlargest(10).index
)

final_plot_data = filtered_apps_df[filtered_apps_df['Category'].isin(top_10_categories_by_installs)]

category_metrics = final_plot_data.groupby('Category').agg(
    Average_Rating=('Rating', 'mean'),
    Total_Reviews=('Reviews', 'sum')
).reset_index()

melted_metrics = category_metrics.melt(id_vars=['Category'], var_name='Metric', value_name='Value')

# Step 2: Grouped bar chart
fig_grouped_bar = px.bar(
    melted_metrics,
    x='Category', y='Value', color='Metric', barmode='group',
    title='Avg Rating & Total Reviews for Top Categories (Filtered)',
    labels={'Category': 'App Category', 'Value': 'Value', 'Metric': 'Metric'},
    color_discrete_map={
        'Average_Rating': px.colors.sequential.Plasma[3],
        'Total_Reviews': px.colors.sequential.Plasma[6]
    },
    width=plot_width, height=plot_height
)
fig_grouped_bar.update_layout(
    plot_bgcolor='black', paper_bgcolor='black', font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

plot_filename = "filtered_category_metrics_grouped_bar.html"
insights_text = (
    "Comparison of average rating and total review count for top 10 app "
    "categories by installs, filtered for January updates, rating >= 4.0, "
    "and size >= 10MB. Visible only 3 PM-5 PM IST."
)

save_plot_as_html(fig_grouped_bar, plot_filename, insights_text)

# Step 3: Time-gate JS (FIXED end hour: 17, not 24)
safe_id = plot_filename.replace('.', '_')
js_code_for_time_check = f"""
<script>
    function checkAndHidePlot_{safe_id}() {{
        const plotContainer = document.getElementById('{plot_filename}');
        if (!plotContainer) return;

        const options = {{ hour: 'numeric', hour12: false, timeZone: 'Asia/Kolkata' }};
        const istHour = parseInt(new Intl.DateTimeFormat('en-US', options).format(new Date()));

        const startTimeHour = 15; // 3 PM IST
        const endTimeHour = 17;   // 5 PM IST (exclusive)

        plotContainer.style.display =
            (istHour >= startTimeHour && istHour < endTimeHour) ? 'block' : 'none';
    }}
    window.addEventListener('load', checkAndHidePlot_{safe_id});
    setInterval(checkAndHidePlot_{safe_id}, 60 * 1000); // re-check every minute
</script>
"""

plot_containers += js_code_for_time_check

# Step 4: Rebuild and re-save the dashboard AFTER adding the new plot + JS
dashboard_full_html = dashboard_html.format(
    plot_width=plot_width, plot_height=plot_height, plots=plot_containers
)

dashboard_path = os.path.join(html_files_paths, "web page.html")
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(dashboard_full_html)

print("Dashboard updated with time-gated grouped bar chart (visible 3-5 PM IST).")

Dashboard updated with time-gated grouped bar chart (visible 3-5 PM IST).


###Task 2


In [53]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# --- Fresh, independent data load ---
t2_apps_df = pd.read_csv('../data/Play Store Data.csv')

# --- Basic cleaning ---
t2_apps_df = t2_apps_df.dropna(subset=['Rating', 'Category', 'Installs'])
t2_apps_df['Installs'] = (
    t2_apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
)
t2_apps_df = t2_apps_df[t2_apps_df['Installs'].str.isnumeric()]
t2_apps_df['Installs'] = t2_apps_df['Installs'].astype(int)

# --- Filter: category must NOT start with A, C, G, or S ---
t2_apps_df = t2_apps_df[~t2_apps_df['Category'].str.upper().str.startswith(('A', 'C', 'G', 'S'))]

# --- Top 5 categories by total installs ---
t2_top5_categories = t2_apps_df.groupby('Category')['Installs'].sum().nlargest(5).index.tolist()
t2_filtered = t2_apps_df[t2_apps_df['Category'].isin(t2_top5_categories)]

# --- Simulate country-level install distribution ---
# (No geo column exists in the dataset — this proportionally spreads each
# category's total installs across a fixed set of countries.)
np.random.seed(42)
t2_countries = ['USA','IND','GBR','DEU','FRA','BRA','CAN','AUS','JPN','CHN',
                 'RUS','ZAF','MEX','ITA','ESP','KOR','IDN','NGA','EGY','ARG']

t2_rows = []
for category in t2_top5_categories:
    total_installs = t2_filtered.loc[t2_filtered['Category'] == category, 'Installs'].sum()
    weights = np.random.dirichlet(np.ones(len(t2_countries)))
    for country, w in zip(t2_countries, weights):
        t2_rows.append({'Category': category, 'Country': country,
                         'Installs': int(total_installs * w)})

t2_geo_df = pd.DataFrame(t2_rows)
t2_geo_df['Highlight'] = np.where(t2_geo_df['Installs'] > 1_000_000, 'Above 1M ⭐', 'Below 1M')

# --- Choropleth (animation_frame lets you flip through the 5 categories) ---
fig_t2 = px.choropleth(
    t2_geo_df,
    locations='Country',
    color='Installs',
    hover_name='Country',
    hover_data={'Category': True, 'Installs': True, 'Highlight': True},
    animation_frame='Category',
    color_continuous_scale=px.colors.sequential.Plasma,
    title='Global Installs by Category (Top 5, excluding A/C/G/S-starting categories)',
    width=800, height=500
)
fig_t2.update_layout(
    paper_bgcolor='black', plot_bgcolor='black', font_color='white',
    geo=dict(bgcolor='black', showframe=False, showcoastlines=True)
)

# --- Save with time-gate: visible ONLY 6 PM – 8 PM IST ---
t2_filename = "task2_choropleth_installs.html"
t2_html_content = pio.to_html(fig_t2, full_html=False, include_plotlyjs='inline')

t2_block = f"""
<div class="plot-container" id="{t2_filename}">
  <div class="plot">{t2_html_content}</div>
  <div class="insights">Global install distribution for the top 5 categories
  (A/C/G/S-starting categories excluded). Countries with over 1M installs are
  flagged in hover text.</div>
</div>
<script>
  function checkTask2Visibility() {{
    const el = document.getElementById("{t2_filename}");
    if (!el) return;
    const istHour = parseInt(new Intl.DateTimeFormat('en-US', {{
        hour: 'numeric', hour12: false, timeZone: 'Asia/Kolkata'
    }}).format(new Date()));
    el.style.display = (istHour >= 18 && istHour < 20) ? 'block' : 'none';
  }}
  window.addEventListener('load', checkTask2Visibility);
  setInterval(checkTask2Visibility, 60000);
</script>
"""

with open(t2_filename, "w", encoding="utf-8") as f:
    f.write(f"<html><body style='background:black;margin:0'>{t2_block}</body></html>")

print("Task 2 saved:", t2_filename)


Task 2 saved: task2_choropleth_installs.html


###Task 3


In [54]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

t3_apps_df = pd.read_csv('../data/Play Store Data.csv')
t3_apps_df = t3_apps_df.dropna(subset=['Rating', 'Category', 'Installs', 'Price',
                                        'Android Ver', 'Size', 'Content Rating', 'App'])

# --- Clean numeric columns ---
t3_apps_df['Installs'] = t3_apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
t3_apps_df = t3_apps_df[t3_apps_df['Installs'].str.isnumeric()]
t3_apps_df['Installs'] = t3_apps_df['Installs'].astype(int)

t3_apps_df['Price'] = t3_apps_df['Price'].astype(str).str.replace('$', '', regex=False)
t3_apps_df['Price'] = pd.to_numeric(t3_apps_df['Price'], errors='coerce')
t3_apps_df = t3_apps_df.dropna(subset=['Price'])
t3_apps_df['Revenue'] = t3_apps_df['Price'] * t3_apps_df['Installs']

def t3_convert_size(size):
    size = str(size)
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    return np.nan
t3_apps_df['Size'] = t3_apps_df['Size'].apply(t3_convert_size)

def t3_parse_android_ver(v):
    match = re.search(r'(\d+(\.\d+)?)', str(v))
    return float(match.group(1)) if match else np.nan
t3_apps_df['Android_Ver_Num'] = t3_apps_df['Android Ver'].apply(t3_parse_android_ver)

# --- Apply all filters ---
t3_filtered = t3_apps_df[
    (t3_apps_df['Installs'] >= 10000) &
    (t3_apps_df['Revenue'] >= 10000) &
    (t3_apps_df['Android_Ver_Num'] > 4.0) &
    (t3_apps_df['Size'] >= 15) &
    (t3_apps_df['Content Rating'] == 'Everyone') &
    (t3_apps_df['App'].str.len() <= 30)
]

# --- Top 3 categories by total installs (post-filter) ---
t3_top3_categories = t3_filtered.groupby('Category')['Installs'].sum().nlargest(3).index.tolist()
t3_final = t3_filtered[t3_filtered['Category'].isin(t3_top3_categories)]

t3_summary = t3_final.groupby(['Category', 'Type']).agg(
    Avg_Installs=('Installs', 'mean'),
    Avg_Revenue=('Revenue', 'mean')
).reset_index()

# --- Dual-axis chart ---
fig_t3 = make_subplots(specs=[[{"secondary_y": True}]])

for app_type, color in [('Free', '#00CC96'), ('Paid', '#EF553B')]:
    sub = t3_summary[t3_summary['Type'] == app_type]
    fig_t3.add_trace(
        go.Bar(x=sub['Category'], y=sub['Avg_Installs'], name=f'{app_type} - Avg Installs',
               marker_color=color, opacity=0.7),
        secondary_y=False
    )
    fig_t3.add_trace(
        go.Scatter(x=sub['Category'], y=sub['Avg_Revenue'], name=f'{app_type} - Avg Revenue',
                    mode='lines+markers', line=dict(width=3)),
        secondary_y=True
    )

fig_t3.update_layout(
    title='Avg Installs vs Avg Revenue — Free vs Paid (Top 3 Categories, Filtered)',
    barmode='group', plot_bgcolor='black', paper_bgcolor='black', font_color='white',
    width=800, height=450
)
fig_t3.update_yaxes(title_text="Avg Installs", secondary_y=False)
fig_t3.update_yaxes(title_text="Avg Revenue ($)", secondary_y=True)

# --- Save with time-gate: visible ONLY 1 PM – 2 PM IST ---
t3_filename = "task3_dual_axis_installs_revenue.html"
t3_html_content = pio.to_html(fig_t3, full_html=False, include_plotlyjs='inline')

t3_block = f"""
<div class="plot-container" id="{t3_filename}">
  <div class="plot">{t3_html_content}</div>
  <div class="insights">Avg installs (bars, left axis) vs avg revenue (lines, right axis)
  for Free vs Paid apps across the top 3 filtered categories.</div>
</div>
<script>
  function checkTask3Visibility() {{
    const el = document.getElementById("{t3_filename}");
    if (!el) return;
    const istHour = parseInt(new Intl.DateTimeFormat('en-US', {{
        hour: 'numeric', hour12: false, timeZone: 'Asia/Kolkata'
    }}).format(new Date()));
    el.style.display = (istHour >= 13 && istHour < 14) ? 'block' : 'none';
  }}
  window.addEventListener('load', checkTask3Visibility);
  setInterval(checkTask3Visibility, 60000);
</script>
"""

with open(t3_filename, "w", encoding="utf-8") as f:
    f.write(f"<html><body style='background:black;margin:0'>{t3_block}</body></html>")

print("Task 3 saved:", t3_filename)

Task 3 saved: task3_dual_axis_installs_revenue.html


###Task 4

In [55]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

t4_apps_df = pd.read_csv('../data/Play Store Data.csv')
t4_apps_df = t4_apps_df.dropna(subset=['Rating', 'Category', 'Installs', 'Reviews',
                                        'Last Updated', 'App'])

t4_apps_df['Installs'] = t4_apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
t4_apps_df = t4_apps_df[t4_apps_df['Installs'].str.isnumeric()]
t4_apps_df['Installs'] = t4_apps_df['Installs'].astype(int)
t4_apps_df['Reviews'] = pd.to_numeric(t4_apps_df['Reviews'], errors='coerce')
t4_apps_df['Last Updated'] = pd.to_datetime(t4_apps_df['Last Updated'], errors='coerce')
t4_apps_df = t4_apps_df.dropna(subset=['Last Updated', 'Reviews'])

# --- Filters ---
t4_apps_df = t4_apps_df[~t4_apps_df['App'].str.lower().str.startswith(('x', 'y', 'z'))]
t4_apps_df = t4_apps_df[t4_apps_df['Category'].str.upper().str.startswith(('E', 'C', 'B'))]
t4_apps_df = t4_apps_df[t4_apps_df['Reviews'] > 500]
t4_apps_df = t4_apps_df[~t4_apps_df['App'].str.upper().str.contains('S')]

# --- Monthly installs per category ---
t4_apps_df['Month'] = t4_apps_df['Last Updated'].dt.to_period('M').dt.to_timestamp()
t4_monthly = t4_apps_df.groupby(['Category', 'Month'])['Installs'].sum().reset_index()
t4_monthly = t4_monthly.sort_values(['Category', 'Month'])

# --- MoM % growth per category ---
t4_monthly['Pct_Growth'] = t4_monthly.groupby('Category')['Installs'].pct_change() * 100

# --- Category name translations for display ---
t4_translate = {'Beauty': 'सौंदर्य (Beauty)', 'Business': 'வணிகம் (Business)',
                 'Dating': 'Dating (Partnersuche)'}
t4_monthly['Category_Display'] = t4_monthly['Category'].replace(t4_translate)

# --- Build line chart with shaded high-growth segments ---
fig_t4 = go.Figure()
t4_colors = px.colors.qualitative.Plotly if False else None  # placeholder, using manual cycle below
import plotly.express as px
t4_color_cycle = px.colors.qualitative.Set2

for i, category in enumerate(t4_monthly['Category'].unique()):
    sub = t4_monthly[t4_monthly['Category'] == category].reset_index(drop=True)
    display_name = sub['Category_Display'].iloc[0]
    color = t4_color_cycle[i % len(t4_color_cycle)]

    # base line
    fig_t4.add_trace(go.Scatter(
        x=sub['Month'], y=sub['Installs'], mode='lines+markers',
        name=display_name, line=dict(color=color, width=2)
    ))

    # shade segments where MoM growth > 20%
    for j in range(1, len(sub)):
        if sub.loc[j, 'Pct_Growth'] > 20:
            fig_t4.add_trace(go.Scatter(
                x=[sub.loc[j-1, 'Month'], sub.loc[j, 'Month']],
                y=[sub.loc[j-1, 'Installs'], sub.loc[j, 'Installs']],
                fill='tozeroy', mode='none',
                fillcolor=color, opacity=0.25,
                showlegend=False, hoverinfo='skip'
            ))

fig_t4.update_layout(
    title='Installs Trend by Category (Shaded = >20% MoM Growth)',
    plot_bgcolor='black', paper_bgcolor='black', font_color='white',
    xaxis_title='Month', yaxis_title='Total Installs',
    width=850, height=450
)

# --- Save with time-gate: visible ONLY 6 PM – 9 PM IST ---
t4_filename = "task4_timeseries_installs_growth.html"
t4_html_content = pio.to_html(fig_t4, full_html=False, include_plotlyjs='inline')

t4_block = f"""
<div class="plot-container" id="{t4_filename}">
  <div class="plot">{t4_html_content}</div>
  <div class="insights">Monthly install trends for Beauty, Business, Dating, and other
  E/C/B categories. Shaded regions mark months where installs grew &gt;20% vs the
  previous month.</div>
</div>
<script>
  function checkTask4Visibility() {{
    const el = document.getElementById("{t4_filename}");
    if (!el) return;
    const istHour = parseInt(new Intl.DateTimeFormat('en-US', {{
        hour: 'numeric', hour12: false, timeZone: 'Asia/Kolkata'
    }}).format(new Date()));
    el.style.display = (istHour >= 18 && istHour < 21) ? 'block' : 'none';
  }}
  window.addEventListener('load', checkTask4Visibility);
  setInterval(checkTask4Visibility, 60000);
</script>
"""

with open(t4_filename, "w", encoding="utf-8") as f:
    f.write(f"<html><body style='background:black;margin:0'>{t4_block}</body></html>")

print("Task 4 saved:", t4_filename)

Task 4 saved: task4_timeseries_installs_growth.html


###Task 5

In [56]:
# Requires TextBlob for sentiment subjectivity (not in source CSVs)
!pip install -q textblob
import nltk
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from textblob import TextBlob
import plotly.express as px
import plotly.io as pio

t5_apps_df = pd.read_csv('../data/Play Store Data.csv')
t5_reviews_df = pd.read_csv('../data/User Reviews.csv')

t5_apps_df = t5_apps_df.dropna(subset=['Rating', 'Category', 'Installs', 'Size',
                                        'Reviews', 'App'])
t5_reviews_df = t5_reviews_df.dropna(subset=['Translated_Review', 'App'])

# --- Clean apps_df ---
t5_apps_df['Installs'] = t5_apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
t5_apps_df = t5_apps_df[t5_apps_df['Installs'].str.isnumeric()]
t5_apps_df['Installs'] = t5_apps_df['Installs'].astype(int)
t5_apps_df['Reviews'] = pd.to_numeric(t5_apps_df['Reviews'], errors='coerce')

def t5_convert_size(size):
    size = str(size)
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    return np.nan
t5_apps_df['Size'] = t5_apps_df['Size'].apply(t5_convert_size)
t5_apps_df = t5_apps_df.dropna(subset=['Size', 'Reviews'])

# --- Compute sentiment subjectivity per review, then average per app ---
t5_reviews_df['Subjectivity'] = t5_reviews_df['Translated_Review'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)
t5_app_subjectivity = t5_reviews_df.groupby('App')['Subjectivity'].mean().reset_index()

t5_merged = pd.merge(t5_apps_df, t5_app_subjectivity, on='App', how='inner')

# --- Apply all filters ---
t5_target_categories = ['GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION',
                         'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENTS']

t5_final = t5_merged[
    (t5_merged['Rating'] > 3.5) &
    (t5_merged['Category'].isin(t5_target_categories)) &
    (t5_merged['Reviews'] > 500) &
    (~t5_merged['App'].str.upper().str.contains('S')) &
    (t5_merged['Subjectivity'] > 0.5) &
    (t5_merged['Installs'] > 50000)
]

# --- Translations for display ---
t5_translate = {'BEAUTY': 'सौंदर्य (Beauty)', 'BUSINESS': 'வணிகம் (Business)',
                 'DATING': 'Dating (Partnersuche)'}
t5_final = t5_final.copy()
t5_final['Category_Display'] = t5_final['Category'].replace(t5_translate)

# --- Color map: highlight Game in pink, others default palette ---
t5_categories_display = t5_final['Category_Display'].unique().tolist()
t5_palette = px.colors.qualitative.Set2
t5_color_map = {}
palette_i = 0
for cat in t5_categories_display:
    if cat == 'GAME':
        t5_color_map[cat] = 'hotpink'
    else:
        t5_color_map[cat] = t5_palette[palette_i % len(t5_palette)]
        palette_i += 1

fig_t5 = px.scatter(
    t5_final, x='Size', y='Rating', size='Installs', color='Category_Display',
    color_discrete_map=t5_color_map,
    hover_name='App', hover_data=['Installs', 'Reviews', 'Subjectivity'],
    title='App Size vs Rating (Bubble = Installs) — Game highlighted in Pink',
    size_max=45, width=850, height=500
)
fig_t5.update_layout(plot_bgcolor='black', paper_bgcolor='black', font_color='white')

# --- Save with time-gate: visible ONLY 5 PM – 7 PM IST ---
t5_filename = "task5_bubble_size_rating.html"
t5_html_content = pio.to_html(fig_t5, full_html=False, include_plotlyjs='inline')

t5_block = f"""
<div class="plot-container" id="{t5_filename}">
  <div class="plot">{t5_html_content}</div>
  <div class="insights">Bubble size = installs. Rating &gt; 3.5, reviews &gt; 500,
  install count &gt; 50k, review subjectivity &gt; 0.5. Game category highlighted pink.</div>
</div>
<script>
  function checkTask5Visibility() {{
    const el = document.getElementById("{t5_filename}");
    if (!el) return;
    const istHour = parseInt(new Intl.DateTimeFormat('en-US', {{
        hour: 'numeric', hour12: false, timeZone: 'Asia/Kolkata'
    }}).format(new Date()));
    el.style.display = (istHour >= 17 && istHour < 19) ? 'block' : 'none';
  }}
  window.addEventListener('load', checkTask5Visibility);
  setInterval(checkTask5Visibility, 60000);
</script>
"""

with open(t5_filename, "w", encoding="utf-8") as f:
    f.write(f"<html><body style='background:black;margin:0'>{t5_block}</body></html>")

print("Task 5 saved:", t5_filename)

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Task 5 saved: task5_bubble_size_rating.html


###Task 6

In [57]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

t6_apps_df = pd.read_csv('../data/Play Store Data.csv')
t6_apps_df = t6_apps_df.dropna(subset=['Rating', 'Category', 'Installs', 'App',
                                        'Reviews', 'Size', 'Last Updated'])

t6_apps_df['Installs'] = t6_apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
t6_apps_df = t6_apps_df[t6_apps_df['Installs'].str.isnumeric()]
t6_apps_df['Installs'] = t6_apps_df['Installs'].astype(int)
t6_apps_df['Reviews'] = pd.to_numeric(t6_apps_df['Reviews'], errors='coerce')
t6_apps_df['Last Updated'] = pd.to_datetime(t6_apps_df['Last Updated'], errors='coerce')

def t6_convert_size(size):
    size = str(size)
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    return np.nan
t6_apps_df['Size'] = t6_apps_df['Size'].apply(t6_convert_size)
t6_apps_df = t6_apps_df.dropna(subset=['Size', 'Reviews', 'Last Updated'])

# --- Filters ---
t6_apps_df = t6_apps_df[t6_apps_df['Rating'] >= 4.2]
t6_apps_df = t6_apps_df[~t6_apps_df['App'].str.contains(r'\d', regex=True)]
t6_apps_df = t6_apps_df[t6_apps_df['Category'].str.upper().str.startswith(('T', 'P'))]
t6_apps_df = t6_apps_df[t6_apps_df['Reviews'] > 1000]
t6_apps_df = t6_apps_df[(t6_apps_df['Size'] >= 20) & (t6_apps_df['Size'] <= 80)]

# --- Monthly totals + cumulative installs per category ---
t6_apps_df['Month'] = t6_apps_df['Last Updated'].dt.to_period('M').dt.to_timestamp()
t6_monthly = t6_apps_df.groupby(['Category', 'Month'])['Installs'].sum().reset_index()
t6_monthly = t6_monthly.sort_values(['Category', 'Month'])
t6_monthly['Cumulative_Installs'] = t6_monthly.groupby('Category')['Installs'].cumsum()
t6_monthly['Pct_Growth'] = t6_monthly.groupby('Category')['Installs'].pct_change() * 100
t6_monthly['High_Growth'] = t6_monthly['Pct_Growth'] > 25

# --- Translated legend labels ---
t6_translate = {'Travel & Local': 'Voyages et Local (FR)',
                 'Productivity': 'Productividad (ES)',
                 'Photography': '写真 (JA)'}
t6_monthly['Category_Display'] = t6_monthly['Category'].replace(t6_translate)

# --- Stacked area chart (base layer) ---
fig_t6 = px.area(
    t6_monthly, x='Month', y='Cumulative_Installs', color='Category_Display',
    title='Cumulative Installs Over Time by Category (Stacked)',
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=850, height=480
)

# --- Overlay markers on months with >25% MoM growth (color-intensity highlight) ---
t6_highlight = t6_monthly[t6_monthly['High_Growth']]
fig_t6.add_trace(go.Scatter(
    x=t6_highlight['Month'], y=t6_highlight['Cumulative_Installs'],
    mode='markers', marker=dict(size=12, color='yellow', symbol='star',
                                 line=dict(color='black', width=1)),
    name='>25% MoM growth', hoverinfo='text',
    text=t6_highlight['Category_Display'] + ' — ' + t6_highlight['Pct_Growth'].round(1).astype(str) + '%'
))

fig_t6.update_layout(
    plot_bgcolor='black', paper_bgcolor='black', font_color='white',
    xaxis_title='Month', yaxis_title='Cumulative Installs'
)

# --- Save with time-gate: visible ONLY 4 PM – 6 PM IST ---
t6_filename = "task6_stacked_area_cumulative_installs.html"
t6_html_content = pio.to_html(fig_t6, full_html=False, include_plotlyjs='inline')

t6_block = f"""
<div class="plot-container" id="{t6_filename}">
  <div class="plot">{t6_html_content}</div>
  <div class="insights">Cumulative installs by category (T/P-starting only). Gold stars
  mark months where a category's installs grew &gt;25% vs the previous month.
  Legend shows translated names for Travel &amp; Local, Productivity, Photography.</div>
</div>
<script>
  function checkTask6Visibility() {{
    const el = document.getElementById("{t6_filename}");
    if (!el) return;
    const istHour = parseInt(new Intl.DateTimeFormat('en-US', {{
        hour: 'numeric', hour12: false, timeZone: 'Asia/Kolkata'
    }}).format(new Date()));
    el.style.display = (istHour >= 16 && istHour < 18) ? 'block' : 'none';
  }}
  window.addEventListener('load', checkTask6Visibility);
  setInterval(checkTask6Visibility, 60000);
</script>
"""

with open(t6_filename, "w", encoding="utf-8") as f:
    f.write(f"<html><body style='background:black;margin:0'>{t6_block}</body></html>")

print("Task 6 saved:", t6_filename)

Task 6 saved: task6_stacked_area_cumulative_installs.html
